In [12]:
from shutil import rmtree
from pathlib import Path

import matplotlib.pyplot as plt

from vot_utils.data import DATA_DIRECTORY, RESULTS_DIRECTORY

In [13]:
import torch

from torch import is_tensor
from torch import tensor
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader

In [14]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [15]:
from functools import partial

from references.transfusion_pytorch.transfusion_pytorch.transfusion import (
    Attention,
    FeedForward,
    Transformer,
    default,
    exists,
    cast_tuple,
    default_to_modality_shape_fn,
    identity,
    add_temp_batch_dim,
    char_tokenize,
    decode_chars,
    append_dims,
    ModalityInfo,
    get_model_output_to_flow_fn,
    pack_one_with_inverse,
    divisible_by,
    eval_decorator,
    rearrange,
    repeat,
    typecheck,
    min_p_filter,
    max_neg_value,
    gumbel_sample,
    cat
)

import torch.nn.functional as F

from torch import nn
from torch.nn import ModuleList
from torch.nn import Linear

from axial_positional_embedding import ContinuousAxialPositionalEmbedding
from rotary_embedding_torch import RotaryEmbedding, apply_rotary_emb
from torchdiffeq import odeint
from ema_pytorch import EMA

from beartype import beartype
from beartype.door import is_bearable

In [16]:
class Transfusion(nn.Module):

    def __init__(
        self,
        num_text_tokens,
        transformer,
        dim_latent=None,
        model_output_clean=True,
        channel_first_latent=False,
        modality_default_shape=None,
        modality_encoder=None,
        modality_decoder=None,
        add_pos_emb=False,
        modality_num_dim=None,
        velocity_consistency_loss_weight=0.1,
        reconstruction_loss_weight=0,
        to_modality_shape_fn=default_to_modality_shape_fn,
        fallback_to_default_shape_if_invalid=False,
        modality_encoder_decoder_requires_batch_dim=True,
        pre_post_transformer_enc_dec=None,
        ignore_index=-1,
        flow_loss_weight=1.0,
        text_loss_weight=1.0,
        odeint_kwargs=dict(atol=1e-5, rtol=1e-5, method="midpoint"),
        eps=1e-2,
        prob_uncond=0.1,
    ):
        super().__init__()

        if isinstance(transformer, dict):
            transformer = Transformer(**transformer).to(device)

        self.transformer = transformer

        self.dim = transformer.dim
        dim_latent = default(dim_latent, self.dim)

        self.dim_latents = cast_tuple(dim_latent)

        self.num_modalities = len(self.dim_latents)
        self.channel_first_latent = cast_tuple(
            channel_first_latent, self.num_modalities
        )
        self.to_modality_shape_fn = cast_tuple(
            to_modality_shape_fn, self.num_modalities
        )

        if not exists(modality_default_shape) or is_bearable(
            modality_default_shape, tuple[int, ...]
        ):
            modality_default_shape = (modality_default_shape,) * self.num_modalities

        self.modality_default_shape = modality_default_shape

        self.modality_num_dim = cast_tuple(modality_num_dim, self.num_modalities)
        self.add_pos_emb = cast_tuple(add_pos_emb, self.num_modalities)

        self.pos_emb_mlp = ModuleList([])

        for modality_add_pos_emb, modality_ndim in zip(
            self.add_pos_emb, self.modality_num_dim
        ):
            if not modality_add_pos_emb:
                self.pos_emb_mlp.append(None)
                continue

            pos_generating_mlp = ContinuousAxialPositionalEmbedding(
                dim=self.dim,
                num_axial_dims=modality_ndim,
            )

            self.pos_emb_mlp.append(pos_generating_mlp)

        modality_encoder = cast_tuple(
            modality_encoder, 1 if exists(modality_encoder) else self.num_modalities
        )
        modality_decoder = cast_tuple(
            modality_decoder, 1 if exists(modality_decoder) else self.num_modalities
        )

        self.modality_encoder = ModuleList(modality_encoder)
        self.modality_decoder = ModuleList(modality_decoder)

        self.maybe_add_temp_batch_dim = (
            add_temp_batch_dim
            if modality_encoder_decoder_requires_batch_dim
            else identity
        )

        self.num_text_tokens = num_text_tokens

        num_text_special_ids = 3
        self.sos_id, self.eos_id, self.null_text_id = (
            num_text_tokens,
            (num_text_tokens + 1),
            (num_text_tokens + 2),
        )

        num_modality_special_ids = self.num_modalities * 2
        som_eom_tensor = (
            torch.arange(num_modality_special_ids)
            + num_text_tokens
            + num_text_special_ids
        )
        som_eom_tensor = rearrange(
            som_eom_tensor, "(start_end m) -> start_end m", start_end=2
        )
        self.som_ids, self.eom_ids = som_eom_tensor.tolist()

        meta_token_offset = (
            num_text_tokens + num_text_special_ids + num_modality_special_ids
        )
        self.meta_id = meta_token_offset
        num_meta_tokens = 128 + 1

        self.char_tokenizer = partial(char_tokenize, offset=meta_token_offset + 1)
        self.decode_chars = partial(decode_chars, offset=meta_token_offset + 1)

        pre_post_transformer_enc_dec = cast_tuple(
            pre_post_transformer_enc_dec, self.num_modalities
        )

        latent_to_model_projs = []
        model_to_latent_projs = []

        for (
            dim_latent,
            enc_dec,
        ) in zip(self.dim_latents, pre_post_transformer_enc_dec):
            pre_attend_enc, post_attend_dec = default(enc_dec, (None, None))

            latent_to_model_proj = (
                Linear(dim_latent, self.dim)
                if dim_latent != self.dim
                else nn.Identity()
            )
            model_to_latent_proj = Linear(self.dim, dim_latent, bias=False)

            latent_to_model_projs.append(default(pre_attend_enc, latent_to_model_proj))
            model_to_latent_projs.append(default(post_attend_dec, model_to_latent_proj))

        self.latent_to_model_projs = ModuleList(latent_to_model_projs)
        self.model_to_latent_projs = ModuleList(model_to_latent_projs)

        self.rotary_emb = RotaryEmbedding(transformer.dim_head)

        effective_num_text_tokens = (
            num_text_tokens
            + num_text_special_ids
            + num_modality_special_ids
            + num_meta_tokens
        )

        self.text_embed = nn.Embedding(effective_num_text_tokens, self.dim)
        self.to_text_logits = Linear(self.dim, effective_num_text_tokens, bias=False)
        text_only_mask = torch.arange(effective_num_text_tokens) < num_text_tokens
        self.register_buffer("text_only_logits_mask", text_only_mask, persistent=False)

        self.ignore_index = ignore_index
        self.flow_loss_weight = flow_loss_weight
        self.text_loss_weight = text_loss_weight
        self.velocity_consistency_loss_weight = velocity_consistency_loss_weight
        self.has_recon_loss = reconstruction_loss_weight > 0.0
        self.reconstruction_loss_weight = reconstruction_loss_weight
        self.model_output_clean = model_output_clean
        self.eps = eps
        self.odeint_fn = partial(odeint, **odeint_kwargs)
        self.prob_uncond = prob_uncond
        self.register_buffer("zero", tensor(0.0), persistent=False)

    @property
    def device(self):
        return next(self.parameters()).device

    
    @torch.no_grad()
    @eval_decorator
    @typecheck
    def generate_text_only(
        self,
        prompt,
        seq_len,
        temperature = 1.5,
        min_p = 0.1,
    ):

        prompt_seq_len, out = prompt.shape[-1], prompt.clone()
        sample_num_times = max(0, seq_len - prompt_seq_len)

        for _ in range(sample_num_times):
            logits = self.forward(out, return_loss = False)
            logits = logits[:, -1]

            logits = min_p_filter(logits, min_p = min_p)

            logits.masked_fill_(~self.text_only_logits_mask, max_neg_value(logits))

            sample = gumbel_sample(logits, temperature = temperature, dim = -1)

            out = cat((out, sample), dim = -1)

        return out[..., prompt_seq_len:]

    
    @typecheck
    def forward(
        self,
        text,
        return_loss=True,
        return_embed=False,
        cache=None,
        return_hiddens = False,
        return_kv_cache = False
    ):

        device = self.device
        text = text.to(device)

        if return_loss:
            text, labels = text[:, :-1], text[:, 1:]

        # embed text

        text = text.masked_fill(text == -1, 0)
        tokens = self.text_embed(text)

        # rotary

        seq_len = tokens.shape[-2]
        pos = torch.arange(seq_len, device = device)

        rotary_emb = self.rotary_emb(pos)

        # attention

        transformer_out = self.transformer(
            tokens,
            rotary_emb = rotary_emb,
            causal_mask = True,
            cache = cache,
            return_kv_cache = return_kv_cache,
            return_hiddens = True
        )

        embed, hiddens, *maybe_kv_cache = transformer_out
        kv_cache = maybe_kv_cache[0] if return_kv_cache else None

        # text unembedding

        logits = self.to_text_logits(embed)

        if not return_loss:
            ret = (logits,)

            if return_kv_cache:
                ret = (*ret, kv_cache)

            if return_hiddens:
                ret = (*ret, hiddens)

            return ret[0] if len(ret) == 1 else ret

        logits = logits.masked_fill(~self.text_only_logits_mask, max_neg_value(logits))

        loss = F.cross_entropy(
            rearrange(logits, 'b n l -> b l n'),
            labels,
            ignore_index = self.ignore_index
        )

        if not return_hiddens:
            return loss

        return loss, hiddens


#### Run Training loop

In [17]:
import os
import math
import gzip
import random
import numpy as np

import torch
from torch.optim import Adam
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

# from transfusion_pytorch import Transfusion

# constants

NUM_BATCHES = int(1e5)
BATCH_SIZE = 4
GRAD_ACCUM_EVERY = 4
LEARNING_RATE = 1e-4
VALIDATE_EVERY = 100
PRIME_LENGTH = 64
GENERATE_EVERY = 500
GENERATE_LENGTH = 256
SEQ_LEN = 256

def cycle(loader):
    while True:
        for data in loader:
            yield data

def decode_token(token):
    return str(chr(max(32, token)))

def decode_tokens(tokens):
    return "".join(list(map(decode_token, tokens)))


In [18]:
with gzip.open(os.path.join(DATA_DIRECTORY, "enwik8", "enwik8.gz")) as file:
    data = np.frombuffer(file.read(int(95e6)), dtype = np.uint8).copy()
    np_train, np_valid = np.split(data, [int(90e6)])
    data_train, data_val = torch.from_numpy(np_train), torch.from_numpy(np_valid)

In [19]:
model = Transfusion(
    num_text_tokens = 256,
    transformer = dict(
        dim = 384,
        depth = 8,
        dim_head = 64,
        heads = 8,
        attn_laser = True
    )
).to(device)


In [20]:
class TextSamplerDataset(Dataset):
    def __init__(self, data, seq_len):
        super().__init__()
        self.data = data
        self.seq_len = seq_len
        self.data_length = data.shape[0]

    def __len__(self):
        return self.data.size(0) // self.seq_len

    def __getitem__(self, index):
        rand_start = torch.randint(0, self.data_length - self.seq_len, (1,))
        full_seq = self.data[rand_start : rand_start + self.seq_len + 1].long()
        return full_seq

train_dataset = TextSamplerDataset(data_train, SEQ_LEN)
val_dataset = TextSamplerDataset(data_val, SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE)

In [10]:
# optimizer

optim = Adam(model.parameters(), lr = LEARNING_RATE)

train_loader = cycle(train_loader)
val_loader = cycle(val_loader)


In [11]:
from tqdm import tqdm

for i in tqdm(range(NUM_BATCHES), mininterval = 10.0, desc = "training"):

    model.train()

    for _ in range(GRAD_ACCUM_EVERY):
        data = next(train_loader)

        loss = model(data.to(device))

        (loss / GRAD_ACCUM_EVERY).backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

    optim.step()
    optim.zero_grad()

    if divisible_by(i, VALIDATE_EVERY):
        model.eval()
        with torch.no_grad():
            valid_data = next(val_loader)
            loss = model(valid_data.to(device))
            print(f'\nvalid loss: {loss.item():.3f}\n')


    if divisible_by(i, GENERATE_EVERY):
        model.eval()

        inp = random.choice(val_dataset)[:PRIME_LENGTH]
        inp = inp.to(device)

        prime = decode_tokens(inp)
        print(f"\nprime: {prime}\n")

        prompt = inp[None, ...]

        sampled = model.generate_text_only(prompt, GENERATE_LENGTH)

        base_decode_output = decode_tokens(sampled[0])

        print(f"\ngenerated: {base_decode_output}\n")
        print(f'loss: {loss.item():.3f}')

training:   0%|          | 0/100000 [00:00<?, ?it/s]


valid loss: 5.058


prime: ]] monk Brother Maynard and is used near the film's conclusion t



training:   0%|          | 1/100000 [00:14<396:47:35, 14.28s/it]


generated: sIÊ  (5ôUmÏÏ·mas]_zëç=Ï kå   Ë[ Ð UÜ{¿- ^úI ³ ¸ ×7 ¡ r È ?2ùn U ù L"HÙ µóh O:¼!·+®Y»ÉAeÃ¡ØÁJ ¦Æªný»ùt:0ñ×ö1ÈÄl ¥iKí[OÔsÅ¸¡¥ Ì«Óo z÷¦ËÜ§[ë «Ù»Ø/3?ñ ÃB ¶$kC ¡,O«®B""z{Ç

loss: 5.058


training:   0%|          | 97/100000 [01:15<18:46:47,  1.48it/s]


valid loss: 2.554



training:   0%|          | 161/100000 [01:56<20:08:10,  1.38it/s]


KeyboardInterrupt: 